[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/04_dnn_keras/04_dnn_keras.ipynb)

# 04. 정형 데이터에 신경망 적용하기 (Keras)

[03_tree_models](../03_tree_models/03_tree_models.ipynb)에서 랜덤 포레스트로 이동 시간 예측
MAE 3.39분, 생존 예측 정확도 78.3%를 얻었습니다. **같은 데이터를 신경망으로 풀어보고
결과를 비교합니다.**

`ml-curriculum`의 [04_neural_networks](../../ml-curriculum/04_neural_networks/04_neural_networks.ipynb)에서
PyTorch로 신경망의 원리(순전파, 역전파, 활성화 함수)를 다뤘고,
[07_tensorflow_practice](../../ml-curriculum/07_tensorflow_practice/07_tensorflow_practice.ipynb)에서
Keras 기본 문법을 익혔습니다. 이 노트북은 그 위에서 **표 데이터에 특유한 부분**에 집중합니다.

- 컬럼이 수백 개인 이미지와 달리, 표 데이터는 **컬럼마다 의미와 단위가 다릅니다**
- 데이터가 수만 건 규모라 **과적합이 훨씬 빨리** 옵니다
- 그래서 **스케일링, Dropout, EarlyStopping**이 선택이 아니라 필수 도구가 됩니다

## 이 노트북의 구성 — 01~03번과 같습니다

**1부에서 택시(회귀)로 신경망을 처음부터 끝까지 만들고, 2부에서 타이타닉(분류)으로 넘어갑니다.**

| | 데이터 | 다루는 것 |
|---|---|---|
| **1부** | `trips` (회귀) | 스케일링이 왜 필요한가 → `Sequential`로 층 쌓기 → 학습 곡선 읽기 → `EarlyStopping` → 트리 모델과 비교 → 저장 |
| **2부** | `titanic` (분류) | 출력층·손실 함수가 어떻게 바뀌는가 → **429건으로 과적합을 극적으로 관찰** → `Dropout` 효과 |

**신경망의 은닉층은 문제 유형과 무관합니다.** 1부에서 만든 구조를 2부에서 거의 그대로 쓰고,
**출력층과 손실 함수만** 바꿉니다. 그 대응표가 이 노트북에서 가장 중요합니다.

## 이 노트북을 읽는 법

- **셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`). 아래쪽 셀은 위쪽 셀에서 만든
  변수를 그대로 쓰기 때문에, 중간부터 실행하면 `NameError`가 납니다.
- **실행 결과는 저장되어 있지 않습니다.** 코드 셀 아래가 비어 있는 것이 정상이고,
  직접 실행해야 표와 그래프가 나타납니다.
- 본문에 적힌 숫자(예: "MAE 8.33분")는 **실행하면 나오는 값**입니다. 글을 읽으면서
  그 숫자가 어느 셀의 출력인지 짚어보면 이해가 빠릅니다.
- 코드는 **그대로 실행만 해도 되지만**, 숫자를 바꿔 다시 실행해보는 것이 가장 좋은 연습입니다.
- **pandas 문법이 막히면** [00_pandas_for_tabular](../00_pandas_for_tabular/00_pandas_for_tabular.ipynb)에
  이 시리즈에서 쓰는 문법만 모아뒀습니다(`pd.to_datetime`, `get_dummies`, `dropna` …). 사전처럼 찾아보세요.
- 낯선 용어는 [glossary.md](../../../glossary.md)에서 찾아보세요.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib
    # Colab에는 tensorflow가 이미 설치되어 있습니다. 로컬이라면: pip install tensorflow

### 준비 셀 — 라이브러리와 한글 폰트

아래 셀은 네 노트북에 공통으로 들어가는 준비 코드입니다. **내용을 이해할 필요는 없고 그냥
실행**하면 됩니다. `numpy`·`pandas`·`matplotlib`·`seaborn`을 불러오고, 그래프의 한글이
깨지지 않게 폰트를 잡고, 결과가 매번 같도록 무작위 시드(`RANDOM_STATE = 42`)를 고정합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

TensorFlow와 Keras를 불러옵니다. **Colab에는 이미 설치되어 있습니다.**
로컬이라면 `pip install tensorflow`가 필요합니다. 시드를 고정해 결과를 최대한 재현되게 합니다.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 결과를 재현할 수 있도록 시드를 고정합니다
tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)

> **주의: 결과가 실행할 때마다 조금씩 달라집니다.** 시드를 고정해도 GPU/멀티스레드 연산의
> 순서가 매번 미묘하게 달라지기 때문입니다. 이 노트북에 적힌 숫자는 **대략적인 기준**으로
> 보시고, 소수점 셋째 자리까지 똑같이 나오지 않아도 정상입니다.
> (완전한 재현이 필요하면 `tf.config.experimental.enable_op_determinism()`을 쓸 수 있지만
> 학습이 느려집니다.)

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

print("trips:", trips.shape)

### 데이터와 전처리 함수

02번 1부에서 만든 `prepare_trips()`를 그대로 씁니다.
**03번 1부와 완전히 같은 데이터·같은 분할**이라, 마지막에 랜덤 포레스트와 숫자를 직접 비교할 수 있습니다.

In [ ]:
def prepare_trips(raw):
    """[회귀] 택시 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    df = raw[(raw["duration"] > 0) & (raw["speed"] < 60)].copy()   # 이상치 제거
    df = df.drop(columns=["pickup", "dropoff",                     # 시각 자체는 weekday/hour로 대체
                          "pickup_zone", "dropoff_zone",           # 범주가 200개 이상이라 제외
                          "speed",                                 # duration으로 계산한 값 → 정답 누출
                          "total"])                                # fare+tip+tolls의 합 → 중복
    df = df.dropna()                                               # 결측치 행 제거
    df = pd.get_dummies(df, columns=["color", "payment",
                                     "pickup_borough", "dropoff_borough"],
                        drop_first=True)                           # 범주형 → 0/1
    X = df.drop(columns="duration")
    y = df["duration"]
    return X, y

---

# 1부. 택시로 회귀 신경망 만들기

`trips` 하나만 봅니다. 03번 1부와 **완전히 같은 데이터, 같은 전처리**를 써서
마지막에 랜덤 포레스트와 성능을 직접 비교할 수 있게 합니다.

| 절 | 내용 |
|---|---|
| 1~2 | 데이터 준비, **스케일링이 정말 필요한지 직접 확인** |
| 3~4 | `Sequential`로 층 쌓기, 파라미터 세기, 문제 유형별 출력층 대응표 |
| 5~6 | 학습 곡선 읽기, `EarlyStopping`으로 최적 시점 잡기 |
| 7~8 | 트리 모델과 비교, 모델 저장 |

## 1. 데이터 준비 — 트리 모델과 다른 두 가지

03번 1부와 같은 전처리를 쓰되, **두 가지를 추가**합니다.

1. **데이터 누출 컬럼 제거** — 03번에서 찾아낸 `fare`, `tip`, `tolls`
2. **스케일링** — 트리 모델에는 없어도 됐지만 신경망에는 필요합니다 (2절에서 확인)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_reg, y_reg = prepare_trips(trips)
X_reg = X_reg.drop(columns=["fare", "tip", "tolls"])   # 03번 1부에서 찾은 데이터 누출

X_train, X_valid, y_train, y_valid = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

# 스케일링 — fit은 학습 데이터에만 (02번의 규칙)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype("float32")
X_valid_s = scaler.transform(X_valid).astype("float32")

y_train_a = y_train.values.astype("float32")
y_valid_a = y_valid.values.astype("float32")

print(f"[회귀] 학습 {X_train_s.shape}  검증 {X_valid_s.shape}")

> **`float32`로 바꾸는 이유**: TensorFlow의 기본 연산 타입이 `float32`입니다. pandas는
> `float64`를 쓰므로 그대로 넣으면 내부에서 변환이 일어나고 경고가 뜰 수 있습니다.
> `get_dummies`가 만든 `bool` 컬럼도 함께 숫자로 바뀝니다.

---

## 2. 스케일링은 정말 필요한가

02번에서 "신경망에는 스케일링이 필수"라고 했습니다. **직접 확인해봅시다.**
먼저 컬럼별 값의 범위를 보면, 사실 이 데이터는 그렇게 극단적이지 않습니다.

In [ ]:
X_train.describe().T[["min", "max"]].round(2)

비교 실험을 위한 **도우미 함수 두 개**를 만듭니다.

- `build_reg_model`: 매번 같은 구조·같은 초기 가중치로 모델을 만듭니다 (조건만 바뀌도록)
- `try_training`: 학습시킨 뒤 검증 MAE를 출력합니다. 학습이 발산해 `NaN`이 나오면 그렇게 표시합니다

In [ ]:
def build_reg_model(n_features, optimizer="adam"):
    """비교 실험용 회귀 모델 — 매번 같은 구조, 같은 초기 가중치"""
    tf.random.set_seed(42)
    model = keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1),
    ])
    model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
    return model


def try_training(X_tr, X_va, tag, optimizer="adam", epochs=40):
    from sklearn.metrics import mean_absolute_error

    model = build_reg_model(X_tr.shape[1], optimizer)
    hist = model.fit(X_tr, y_train_a, validation_data=(X_va, y_valid_a),
                     epochs=epochs, batch_size=128, verbose=0)
    pred = model.predict(X_va, verbose=0).ravel()

    if not np.isfinite(pred).all():
        result = "발산 (NaN)"
    else:
        result = f"{mean_absolute_error(y_valid_a, pred):.4f}"
    print(f"  {tag:22s} 검증 MAE {result:>12s}   최종 학습 손실 {hist.history['loss'][-1]:>12.4g}")

먼저 **Adam** 옵티마이저로, 스케일링한 경우와 안 한 경우를 비교합니다.

In [ ]:
print("[optimizer=adam]")
try_training(X_train.values.astype("float32"), X_valid.values.astype("float32"), "스케일링 없음")
try_training(X_train_s, X_valid_s, "StandardScaler")

**거의 차이가 없습니다.** 스케일링이 필수라더니 왜 그럴까요?

이 데이터의 컬럼은 대부분 원-핫 인코딩으로 만든 **0/1**이고, 나머지도 `distance`(0~37),
`hour`(0~23), `passengers`(0~6)로 **범위가 이미 비슷합니다.** 게다가 **Adam은 파라미터마다
학습률을 자동 조절**하는 옵티마이저라 스케일 차이를 어느 정도 흡수합니다.

**그러면 언제 문제가 되는가?** 두 경우를 만들어봅시다.

In [ ]:
print("[optimizer=SGD(learning_rate=0.01)] — 학습률이 고정된 기본 경사 하강법")
try_training(X_train.values.astype("float32"), X_valid.values.astype("float32"),
             "스케일링 없음", keras.optimizers.SGD(0.01))
try_training(X_train_s, X_valid_s, "StandardScaler", keras.optimizers.SGD(0.01))

**스케일링 없이 SGD를 쓰면 학습이 발산해서 `NaN`이 됩니다.**

값이 큰 컬럼에서 나온 큰 기울기에 고정 학습률을 곱하면 가중치가 과하게 움직입니다.
그러면 다음 스텝에서 더 큰 기울기가 나오고, 이것이 반복되며 값이 무한대로 튑니다.

이제 **컬럼 하나의 단위만 바꿔봅시다.** 거리를 마일 대신 밀리미터로 재면 어떻게 될까요?
값 자체는 같은 정보인데 숫자만 160만 배 커집니다.

In [ ]:
X_train_mm = X_train.copy()
X_valid_mm = X_valid.copy()
X_train_mm["distance_mm"] = X_train["distance"] * 1_609_344   # 마일 -> 밀리미터
X_valid_mm["distance_mm"] = X_valid["distance"] * 1_609_344
X_train_mm = X_train_mm.drop(columns="distance")
X_valid_mm = X_valid_mm.drop(columns="distance")

scaler_mm = StandardScaler()
X_train_mm_s = scaler_mm.fit_transform(X_train_mm).astype("float32")
X_valid_mm_s = scaler_mm.transform(X_valid_mm).astype("float32")

print(f"거리 컬럼 최댓값: {X_train_mm['distance_mm'].max():,.0f} mm")
print()
print("[거리를 밀리미터로 바꾼 데이터, adam]")
try_training(X_train_mm.values.astype("float32"), X_valid_mm.values.astype("float32"), "스케일링 없음")
try_training(X_train_mm_s, X_valid_mm_s, "StandardScaler")

**MAE가 3.5분에서 몇 배로 뛰어오릅니다.** 데이터의 정보량은 하나도 달라지지 않았고
단위만 바꿨는데 모델이 눈에 띄게 망가졌습니다. 스케일링을 적용하면 원래대로 돌아옵니다.

> **얼마나 나빠지는지는 실행할 때마다 다릅니다.** 가중치 초기화 운에 따라 MAE가 5분대에 그치기도 하고
> 20분을 넘기기도 하며, 손실이 발산해 버리는 경우도 있습니다. 숫자를 외우지 말고,
> **스케일링한 쪽은 3.5분 근처에서 안정적인데 안 한 쪽은 실행마다 들쭉날쭉하다**는 점을 보세요.
> 재현성 자체가 없다는 것이 이 실험의 결과입니다.

### 정리

| 상황 | 스케일링 없이 |
|---|---|
| 컬럼 범위가 비슷 + Adam | 큰 문제 없음 |
| **고정 학습률 옵티마이저(SGD)** | **발산 위험** |
| **컬럼 간 단위 차이가 큼** | **거의 확실히 실패** |

**"이번엔 괜찮았다"가 "안 해도 된다"를 뜻하지는 않습니다.** 스케일링은 비용이 거의 없고
실패 시 손실은 큽니다. **신경망에는 항상 적용한다**고 외워두는 편이 낫습니다.

> 트리 모델에 스케일링이 필요 없었던 이유를 다시 짚어보면 대비가 분명합니다. 트리는
> "`distance`가 2.5보다 큰가?"만 묻습니다. 밀리미터로 바꿔도 기준값이 4,023,360으로
> 바뀔 뿐 **분할 결과는 완전히 동일**합니다.

---

## 3. `Sequential`로 신경망 만들기

Keras에서 층을 순서대로 쌓는 가장 간단한 방법입니다.

In [ ]:
tf.random.set_seed(42)

model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),   # 입력: 컬럼 개수만큼
    layers.Dense(64, activation="relu"),          # 은닉층 1
    layers.Dense(32, activation="relu"),          # 은닉층 2
    layers.Dense(16, activation="relu"),          # 은닉층 3
    layers.Dense(1),                              # 출력층: 값 1개 (회귀)
])

model.summary()

### `summary()` 읽기

**Output Shape의 `None`** 은 배치 크기 자리입니다. 몇 개를 한 번에 넣을지는 학습할 때
정해지므로 아직 비어 있습니다.

**Param # 계산**은 간단합니다.

```
파라미터 수 = (입력 개수 × 뉴런 개수) + 뉴런 개수
                 └── 가중치 ──┘        └ 편향 ┘
```

| 층 | 계산 | 파라미터 |
|---|---|---|
| Dense(64) | 13 × 64 + 64 | **896** |
| Dense(32) | 64 × 32 + 32 | **2,080** |
| Dense(16) | 32 × 16 + 16 | **528** |
| Dense(1) | 16 × 1 + 1 | **17** |
| | | **합계 3,521** |

**학습 데이터가 5,068건인데 파라미터가 3,521개입니다.** 데이터 한 건당 파라미터가
0.7개꼴이라 **외우기 아주 쉬운 상태**입니다. 표 데이터에서 과적합이 빨리 오는 이유가 여기 있습니다.

> 층 하나에 뉴런을 몇 개 둘지는 정해진 규칙이 없습니다. 관례적으로 **64 → 32 → 16처럼
> 점점 줄여가는** 형태를 많이 쓰는데, 입력을 점진적으로 압축해 요약한다는 직관 때문입니다.
> 표 데이터에서는 **층 2~3개, 뉴런 수십 개 규모면 충분한 경우가 대부분**입니다.
> 깊고 넓게 만든다고 좋아지지 않고, 과적합만 빨라집니다.

In [ ]:
# 파라미터 개수 직접 확인
n_in = X_train_s.shape[1]
print(f"입력 컬럼 수: {n_in}")
print(f"Dense(64): {n_in} × 64 + 64 = {n_in * 64 + 64}")
print(f"Dense(32): 64 × 32 + 32 = {64 * 32 + 32}")
print(f"Dense(16): 32 × 16 + 16 = {32 * 16 + 16}")
print(f"Dense(1) : 16 × 1 + 1 = {16 * 1 + 1}")
print(f"합계: {model.count_params()}")
print(f"학습 데이터: {len(X_train_s)}건")

---

## 4. 문제 유형에 따라 달라지는 것

**신경망의 은닉층은 문제 유형과 무관합니다.** 달라지는 것은 **출력층과 손실 함수**뿐입니다.
이 표가 이 노트북에서 가장 중요합니다.

1부에서는 **회귀** 줄만 씁니다. **이진 분류** 줄은 2부에서 타이타닉으로 실습합니다.

| 문제 | 출력층 뉴런 수 | 출력층 활성화 | 손실 함수 | 지표 |
|---|---|---|---|---|
| **회귀** | 1 | 없음(`linear`) | `mse` 또는 `mae` | `mae`, `mse` |
| **이진 분류** | 1 | `sigmoid` | `binary_crossentropy` | `accuracy` |
| **다중 분류** (정수 라벨) | 클래스 수 | `softmax` | `sparse_categorical_crossentropy` | `accuracy` |
| **다중 분류** (원-핫 라벨) | 클래스 수 | `softmax` | `categorical_crossentropy` | `accuracy` |

**왜 이렇게 되는가**

- **회귀**: 출력이 어떤 실수든 될 수 있어야 하므로 활성화 함수를 걸지 않습니다.
  `sigmoid`를 걸면 출력이 0~1로 갇혀 이동 시간을 예측할 수 없습니다
- **이진 분류**: `sigmoid`가 어떤 값이든 **0~1 사이 확률**로 눌러줍니다
- **다중 분류**: `softmax`는 클래스별 점수를 **합이 1인 확률 분포**로 바꿉니다

**다중 분류의 두 줄이 특히 헷갈립니다.** 차이는 **정답 라벨의 형태**뿐입니다.

```python
y = [0, 2, 1]                      # 정수 라벨    -> sparse_categorical_crossentropy
y = [[1,0,0], [0,0,1], [0,1,0]]    # 원-핫 라벨   -> categorical_crossentropy
                                   #   (to_categorical(y)로 변환)
```

`sparse_`를 쓰면 변환 단계를 생략할 수 있어 더 간편합니다.

---

## 5. 회귀 신경망 — 이동 시간 예측

`compile`로 학습 방식을 정하고, `fit`으로 학습합니다.

```python
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
```

| 인자 | 역할 |
|---|---|
| `optimizer` | 가중치를 어떻게 갱신할지. **`"adam"`이 거의 항상 무난한 기본값** |
| `loss` | **학습에 실제로 쓰이는 값.** 이것을 줄이는 방향으로 가중치가 움직입니다 |
| `metrics` | 사람이 보려고 계산하는 값. **학습에는 영향을 주지 않습니다** |

`loss`는 `mse`, `metrics`는 `mae`로 두는 조합이 흔합니다. **학습은 큰 오차에 민감한 MSE로 하고,
사람은 해석하기 쉬운 MAE로 확인**하는 것입니다.

In [ ]:
tf.random.set_seed(42)

reg_model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),                   # 회귀: 활성화 함수 없음
])

reg_model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = reg_model.fit(
    X_train_s, y_train_a,
    validation_data=(X_valid_s, y_valid_a),
    epochs=60,
    batch_size=128,
    verbose=0,           # 1로 두면 epoch마다 진행 상황이 출력됩니다
)

print("학습 완료:", len(history.history["loss"]), "epoch")

### `fit`의 주요 인자

| 인자 | 의미 |
|---|---|
| `validation_data` | 매 epoch마다 성능을 잴 데이터. **학습에는 쓰이지 않습니다** |
| `epochs` | 전체 데이터를 몇 번 반복할지 |
| `batch_size` | 가중치를 한 번 갱신할 때 쓰는 데이터 수 (기본 32) |
| `callbacks` | 학습 중간에 개입하는 도구들 (6절) |
| `verbose` | 0=조용히, 1=진행 막대, 2=epoch당 한 줄 |

**`batch_size`의 의미**: 5,068건을 `batch_size=128`로 학습하면 한 epoch에
`5068 / 128 ≈ 40`번 가중치가 갱신됩니다.

- **작으면**: 갱신이 자주 일어나 빠르게 수렴하지만 방향이 불안정합니다
- **크면**: 안정적이고 계산이 효율적이지만 갱신 횟수가 줄어듭니다
- 보통 32 ~ 256을 씁니다

### `history` — 학습 기록

`fit`은 `History` 객체를 돌려주고, `history.history`는 **epoch별 지표가 담긴 딕셔너리**입니다.
`validation_data`를 준 지표에는 **`val_` 접두사**가 붙습니다.

In [ ]:
print("기록된 지표:", list(history.history.keys()))
print()
print(f"첫 epoch  : loss {history.history['loss'][0]:8.3f}  val_loss {history.history['val_loss'][0]:8.3f}")
print(f"마지막     : loss {history.history['loss'][-1]:8.3f}  val_loss {history.history['val_loss'][-1]:8.3f}")

학습 곡선을 그리는 함수를 만들어 두고 이 노트북 내내 씁니다.
`history.history`에 담긴 epoch별 지표를 **학습용·검증용 두 줄**로 겹쳐 그립니다.

In [ ]:
def plot_history(hist, metric="loss", title=None):
    """학습 곡선 그리기 — 이 노트북에서 계속 씁니다"""
    plt.figure(figsize=(9, 5))
    plt.plot(hist.history[metric], label=f"학습 {metric}")
    plt.plot(hist.history[f"val_{metric}"], label=f"검증 val_{metric}")
    plt.xlabel("Epoch")
    plt.ylabel(metric.upper())
    plt.title(title or f"Model {metric.upper()}")
    plt.legend()
    plt.show()


plot_history(history, "mae", "이동 시간 예측 — MAE 변화")

### 학습 곡선을 읽는 법

**이 그래프 하나로 모델의 상태를 진단합니다.**

| 모양 | 진단 | 대응 |
|---|---|---|
| 두 곡선이 함께 내려가는 중 | 정상, **아직 덜 학습됨** | epoch 늘리기 |
| 두 곡선이 나란히 평평 | 수렴 완료 | 모델을 키우거나 피처 추가 |
| **학습은 내려가는데 검증이 올라감** | **과적합** | Dropout, EarlyStopping, 단순화 |
| 둘 다 높은 채로 평평 | **과소적합** | 모델 키우기, 학습률 조정 |
| 곡선이 심하게 요동침 | 학습률이 큼 / 배치가 작음 | 학습률 낮추기 |

**검증 곡선이 최저점을 찍고 올라가기 시작하는 지점이 멈춰야 할 곳**입니다.
03번에서 본 트리 깊이 그래프와 정확히 같은 이야기입니다.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

y_pred_nn = reg_model.predict(X_valid_s, verbose=0).ravel()

val_mae = np.array(history.history["val_mae"])
print(f"최종 epoch의 검증 MAE : {val_mae[-1]:.4f}")
print(f"가장 좋았던 검증 MAE   : {val_mae.min():.4f}  (epoch {val_mae.argmin() + 1})")
print()
print(f"신경망 최종 성능: MAE {mean_absolute_error(y_valid_a, y_pred_nn):.4f}분  "
      f"R² {r2_score(y_valid_a, y_pred_nn):.4f}")

**가장 좋았던 지점이 마지막 epoch이 아닌 경우가 흔합니다.** 중간에 최저점을 찍고 다시 나빠졌다면
지금 손에 든 모델은 **최선의 모델이 아닙니다.** (실행에 따라 최저점이 마지막 근처에 올 수도 있습니다.
그럴 때는 `epochs`를 더 늘려보면 대개 중간에서 꺾입니다.) 이 문제를 해결하는 것이 `EarlyStopping`입니다.

## 6. 과적합 막기 — `EarlyStopping`과 `Dropout`

바로 위에서 "가장 좋았던 지점이 마지막 epoch이 아니다"라는 문제를 봤습니다. 그것을 해결하는
도구가 `EarlyStopping`이고, 과적합 자체를 늦추는 도구가 `Dropout`입니다.

### `EarlyStopping` — 최저점에서 멈추기

```python
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",           # 무엇을 감시할지
    patience=20,                  # 몇 epoch 동안 나아지지 않으면 멈출지
    restore_best_weights=True,    # 가장 좋았던 시점의 가중치로 되돌리기
)
```

**`restore_best_weights=True`가 핵심입니다.** 이 옵션이 없으면 "멈춘 시점"의 가중치가
남는데, 그때는 이미 20 epoch만큼 나빠진 상태입니다. **`patience` 때문에 일부러 더 지켜본
구간을 되돌려 놓아야** 최적 모델을 얻습니다.

`patience`는 너무 작으면 일시적인 출렁임에 속아 일찍 멈추고, 너무 크면 시간을 낭비합니다.
보통 **10~30** 사이에서 정합니다.

### `Dropout` — 학습 중에 뉴런을 무작위로 끄기

```python
layers.Dropout(0.3)     # 학습할 때마다 뉴런의 30%를 무작위로 0으로
```

매번 다른 뉴런이 꺼지므로, 모델은 **특정 뉴런 몇 개에만 의존할 수 없게** 됩니다.
여러 개의 작은 신경망을 학습시켜 평균 내는 효과가 있어, 랜덤 포레스트가 트리를 여럿 모으는
것과 발상이 비슷합니다.

**예측할 때는 자동으로 꺼집니다.** 학습 시에만 동작하므로 따로 신경 쓸 필요가 없습니다.

---

## 7. `EarlyStopping`을 적용해 다시 학습

5절의 모델에 방금 본 두 도구를 적용해 마무리합니다.

In [ ]:
tf.random.set_seed(42)

reg_final = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])

reg_final.compile(optimizer="adam", loss="mse", metrics=["mae"])

hist_final = reg_final.fit(
    X_train_s, y_train_a,
    validation_data=(X_valid_s, y_valid_a),
    epochs=200, batch_size=128,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                             restore_best_weights=True)],
    verbose=0,
)

pred_final = reg_final.predict(X_valid_s, verbose=0).ravel()

print(f"학습한 epoch: {len(hist_final.history['loss'])}")
print(f"MAE {mean_absolute_error(y_valid_a, pred_final):.4f}분   "
      f"R² {r2_score(y_valid_a, pred_final):.4f}")

곡선을 그려 5절의 것과 비교해보세요. **검증 곡선이 올라가기 시작하는 지점에서 멈췄는지**가 관전 포인트입니다.

In [ ]:
plot_history(hist_final, "mae", "이동 시간 예측 — EarlyStopping 적용")

---

## 8. 트리 모델 vs 신경망 (회귀)

03번 1부에서 같은 데이터를 랜덤 포레스트로 풀었습니다. **같은 전처리, 같은 분할**이므로
숫자를 그대로 비교할 수 있습니다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# 03번 1부에서 GridSearchCV로 찾은 설정
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5,
                               random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
rf_reg_pred = rf_reg.predict(X_valid)

print("=== 회귀: 이동 시간 예측 ===")
print(f"  기준선(평균)  MAE {mean_absolute_error(y_valid, np.full(len(y_valid), y_train.mean())):.4f}분")
print(f"  랜덤 포레스트  MAE {mean_absolute_error(y_valid, rf_reg_pred):.4f}분  R² {r2_score(y_valid, rf_reg_pred):.4f}")
print(f"  신경망        MAE {mean_absolute_error(y_valid_a, pred_final):.4f}분  R² {r2_score(y_valid_a, pred_final):.4f}")

**두 모델의 성능이 사실상 같습니다.** MAE 차이가 0.05분(3초) 수준이고, **이 정도는 실행할
때마다 순위가 뒤집히는 폭**입니다. 여러분이 실행하면 반대 순서가 나올 수도 있습니다.

들인 노력은 전혀 다릅니다. 랜덤 포레스트는 `fit()` 한 줄이었고, 신경망은 스케일링·층 설계·
`EarlyStopping`이 전부 필요했습니다. **자세한 비교는 2부 마지막에서** 분류 결과까지 함께 봅니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_valid, rf_reg_pred, alpha=0.3, s=10, label="랜덤 포레스트")
axes[0].scatter(y_valid, pred_final, alpha=0.3, s=10, label="신경망")
lims = [0, y_valid.max()]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("실제 (분)")
axes[0].set_ylabel("예측 (분)")
axes[0].set_title("실제 vs 예측")
axes[0].legend()

err = pd.DataFrame({
    "랜덤 포레스트": np.abs(y_valid.values - rf_reg_pred),
    "신경망": np.abs(y_valid_a - pred_final),
})
sns.boxplot(data=err, ax=axes[1])
axes[1].set_ylabel("절대 오차 (분)")
axes[1].set_title("오차 분포")

plt.tight_layout()
plt.show()

---

## 9. 모델 저장하고 불러오기

In [ ]:
import os

os.makedirs("../../../models", exist_ok=True)   # 프로젝트 루트의 models/
path = "../../../models/trip_duration_dnn.keras"

reg_final.save(path)

loaded = keras.models.load_model(path)
loaded_pred = loaded.predict(X_valid_s, verbose=0).ravel()

print(f"저장 위치: {path}")
print(f"원본 모델 MAE  : {mean_absolute_error(y_valid_a, pred_final):.6f}")
print(f"불러온 모델 MAE: {mean_absolute_error(y_valid_a, loaded_pred):.6f}")
print("두 값이 같으면 정상적으로 복원된 것입니다.")

**1부 끝.** 회귀 신경망 하나를 만들고, 저장하고, 트리 모델과 비교까지 했습니다.

이제 타이타닉으로 넘어갑니다. **은닉층은 그대로 두고 출력층과 손실 함수만 바꾸면** 분류가
된다는 것, 그리고 **데이터가 429건뿐일 때 과적합이 얼마나 빨리 오는지**를 봅니다.

---

# 2부. 타이타닉으로 분류 신경망 만들기

4절의 대응표에서 **이진 분류** 줄을 실제로 써봅니다. 1부와 달라지는 것은 **딱 두 줄**입니다.

| | 1부 (회귀) | 2부 (이진 분류) |
|---|---|---|
| 출력층 | `Dense(1)` | `Dense(1, activation="sigmoid")` |
| 손실 함수 | `mse` | `binary_crossentropy` |
| 지표 | `mae` | `accuracy` |

은닉층(`Dense(64) → Dense(32)`)은 그대로입니다.

**여기서 과적합이 훨씬 극적으로 나타납니다.** 학습 데이터가 5,068건에서 **429건**으로 줄기
때문입니다. 1부에서는 60 epoch을 돌려도 완만했던 곡선이, 여기서는 10 epoch 만에 꺾입니다.

## 9. 타이타닉 준비

In [ ]:
def prepare_titanic(raw):
    """[분류] 타이타닉 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    q1, q3 = raw["fare"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    df = raw[(raw["fare"] >= lower) & (raw["fare"] <= upper)].copy()  # 이상치 제거
    df = df.drop(columns=["alive",                                   # survived와 같은 정보 → 정답 누출
                          "class", "embark_town",                    # pclass/embarked와 중복
                          "deck",                                    # 결측치가 77%
                          "adult_male"])                             # who와 중복
    df = df.dropna()
    df = pd.get_dummies(df, columns=["sex", "embarked", "who"], drop_first=True)
    return df.drop(columns="survived"), df["survived"]


titanic = sns.load_dataset("titanic")
X_clf, y_clf = prepare_titanic(titanic)

Xc_train, Xc_valid, yc_train, yc_valid = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_STATE, stratify=y_clf
)

# 신경망이므로 여기서도 스케일링은 필수입니다
scaler_c = StandardScaler()
Xc_train_s = scaler_c.fit_transform(Xc_train).astype("float32")
Xc_valid_s = scaler_c.transform(Xc_valid).astype("float32")

yc_train_a = yc_train.values.astype("float32")
yc_valid_a = yc_valid.values.astype("float32")

print(f"[분류] 학습 {Xc_train_s.shape}  검증 {Xc_valid_s.shape}")

---

---

## 10. 대책 없이 학습하면 — 과적합 관찰

출력층과 손실 함수만 바꾼 모델을 **아무 대책 없이** 150 epoch 돌려봅니다.

In [ ]:
tf.random.set_seed(42)

clf_plain = keras.Sequential([
    layers.Input(shape=(Xc_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),      # 이진 분류: sigmoid
])

clf_plain.compile(optimizer="adam",
                  loss="binary_crossentropy",   # 이진 분류: BCE
                  metrics=["accuracy"])

hist_plain = clf_plain.fit(
    Xc_train_s, yc_train_a,
    validation_data=(Xc_valid_s, yc_valid_a),
    epochs=150, batch_size=32, verbose=0,
)

print("완료")

손실과 정확도를 나란히 그려봅시다. **학습 곡선과 검증 곡선이 벌어지는 지점**을 찾는 것이 목적입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(hist_plain.history[metric], label="학습")
    ax.plot(hist_plain.history[f"val_{metric}"], label="검증")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric)
    ax.set_title(f"Model {metric}")
    ax.legend()

plt.tight_layout()
plt.show()

그래프에서 눈으로 읽은 것을 숫자로 확인합니다.

In [ ]:
h = hist_plain.history
best_epoch = int(np.argmin(h["val_loss"])) + 1

print(f"검증 손실 최저점 : epoch {best_epoch}, val_loss {min(h['val_loss']):.4f}")
print(f"마지막 epoch     : val_loss {h['val_loss'][-1]:.4f}")
print()
print(f"마지막 학습 정확도 : {h['accuracy'][-1]:.4f}")
print(f"마지막 검증 정확도 : {h['val_accuracy'][-1]:.4f}")
print(f"최고 검증 정확도   : {max(h['val_accuracy']):.4f} (epoch {int(np.argmax(h['val_accuracy'])) + 1})")

**전형적인 과적합입니다.**

- 검증 손실은 **10 epoch 근처에서 최저점**을 찍고 이후 계속 올라갑니다
- 학습 정확도는 90%까지 오르는데 검증 정확도는 76% 근처에 머뭅니다
- 429건을 3,000개 가까운 파라미터로 학습하니 당연한 결과입니다

**주목할 점: 검증 손실은 나빠지는데 검증 정확도는 크게 안 떨어집니다.** 손실이 더 민감한
지표이기 때문입니다. 모델이 "생존 확률 0.6"이라고 하던 것을 "0.95"라고 확신하게 되면,
예측 자체는 같아서 정확도는 그대로지만 **틀렸을 때의 손실은 훨씬 커집니다.**
그래서 **과적합 감시는 `val_loss`로 하는 것이 표준**입니다.

---

## 11. `Dropout`과 `EarlyStopping` 적용

1부 6절에서 본 두 도구를 그대로 씁니다. 데이터가 적을수록 효과가 큽니다.

In [ ]:
tf.random.set_seed(42)

clf_model = keras.Sequential([
    layers.Input(shape=(Xc_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

clf_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

hist_clf = clf_model.fit(
    Xc_train_s, yc_train_a,
    validation_data=(Xc_valid_s, yc_valid_a),
    epochs=300,                  # 넉넉히 주고 EarlyStopping에 맡깁니다
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

print(f"300 epoch을 요청했지만 {len(hist_clf.history['loss'])} epoch에서 멈췄습니다")
print(f"가장 좋았던 epoch: {int(np.argmin(hist_clf.history['val_loss'])) + 1}")

대책을 적용한 곡선입니다. 회색 세로선이 **검증 손실 최저점**이고, `restore_best_weights=True`이므로
최종 모델은 이 지점의 가중치를 씁니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(hist_clf.history[metric], label="학습")
    ax.plot(hist_clf.history[f"val_{metric}"], label="검증")
    ax.axvline(np.argmin(hist_clf.history["val_loss"]), color="gray",
               linestyle="--", linewidth=1)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric)
    ax.set_title(f"Dropout + EarlyStopping — {metric}")
    ax.legend()

plt.tight_layout()
plt.show()

대책이 있을 때와 없을 때의 정확도를 비교합니다.
`sigmoid` 출력은 **0~1 사이의 확률**이라, 0.5를 기준으로 잘라 0/1 예측으로 바꿉니다.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

proba = clf_model.predict(Xc_valid_s, verbose=0).ravel()
pred = (proba >= 0.5).astype(int)      # sigmoid 출력은 확률 -> 0.5 기준으로 자르기

print(f"대책 없음        : {accuracy_score(yc_valid_a, (clf_plain.predict(Xc_valid_s, verbose=0).ravel() >= 0.5).astype(int)):.4f}")
print(f"Dropout + EarlyStopping: {accuracy_score(yc_valid_a, pred):.4f}")
print()
print(classification_report(yc_valid_a, pred, target_names=["사망", "생존"]))

**학습이 300 epoch가 아니라 30~40 epoch 근처에서 멈추고, 정확도도 올랐습니다.**
그래프에서도 두 곡선이 훨씬 오래 붙어 있습니다.

> #### 그 밖의 과적합 대책
>
> - **`BatchNormalization`**: 층의 출력을 정규화해 학습을 안정시킵니다. 부수적으로 규제 효과도
>   있습니다. `Dense` → `BatchNormalization` → 활성화 순으로 넣는 것이 일반적입니다
> - **가중치 규제**: `layers.Dense(64, kernel_regularizer=keras.regularizers.l2(0.01))`
> - **모델 축소**: 표 데이터에서는 **이것이 가장 효과적일 때가 많습니다.**
>   층과 뉴런을 줄이는 것만으로 과적합이 크게 줄어듭니다
> - **`ModelCheckpoint`**: 가장 좋은 시점의 모델을 파일로 저장합니다
>   ```python
>   keras.callbacks.ModelCheckpoint("best.keras", monitor="val_loss", save_best_only=True)
>   ```

---

## 12. 트리 모델 vs 신경망 (분류), 그리고 총평

1부에서 회귀를 비교했습니다. 분류도 같은 방식으로 03번 2부의 랜덤 포레스트와 맞대봅니다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 03번 2부에서 GridSearchCV로 찾은 설정
rf_clf = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=5,
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xc_train, yc_train)

print("=== 분류: 생존 예측 ===")
print(f"  기준선(전부 사망) 정확도 {(yc_valid == 0).mean():.4f}")
print(f"  랜덤 포레스트     정확도 {accuracy_score(yc_valid, rf_clf.predict(Xc_valid)):.4f}")
print(f"  신경망           정확도 {accuracy_score(yc_valid_a, pred):.4f}")

### 결과 해석

**두 모델의 성능이 사실상 같습니다.** 회귀는 MAE 차이가 0.05분(3초) 수준이고, 분류는
184건 중 2~3명 차이입니다. **이 정도는 실행할 때마다 순위가 뒤집히는 폭**이라
"어느 쪽이 이겼다"고 말할 수 없습니다. 여러분이 실행하면 이 노트북에 적힌 것과
반대 순서가 나올 수도 있습니다.

주목할 점은 **들인 노력이 전혀 다르다**는 것입니다.

| | 랜덤 포레스트 | 신경망 |
|---|---|---|
| 전처리 | 인코딩만 | 인코딩 + **스케일링 필수** |
| 구조 결정 | 없음 | 층 수, 뉴런 수, 활성화 함수 |
| 학습 설정 | 없음 | 옵티마이저, 배치 크기, epoch 수 |
| 과적합 대책 | `max_depth` 정도 | **Dropout, EarlyStopping 필수** |
| 결과 재현 | `random_state`로 완전 재현 | 완전히 같지는 않음 |

**손이 몇 배로 갔는데 얻은 것이 없습니다.** 이것이 표 데이터의 일반적인 양상입니다. 이미지·음성·텍스트에서는 딥러닝이 압도적이지만,
표 데이터에서는 트리 계열(특히 XGBoost, LightGBM 같은 부스팅)이 여전히 강력합니다.

**왜 그럴까요?**

- **신경망은 입력에 구조가 있다고 가정합니다.** 이미지에서 옆 픽셀은 관련이 있고, 문장에서
  앞 단어는 뒤 단어와 이어집니다. **표 데이터의 컬럼 순서에는 그런 의미가 없습니다.**
  `distance` 옆에 `hour`가 있는 것은 그냥 우연입니다
- **트리는 "이 값보다 큰가"라는 계단식 판단을 자연스럽게 합니다.** "거리가 5마일을 넘으면
  갑자기 고속도로를 탄다" 같은 규칙을 한 번의 분할로 표현합니다. 신경망이 같은 것을
  배우려면 여러 층을 거쳐 근사해야 합니다
- **데이터 양.** 신경망은 데이터가 많을수록 유리한데, 5,068건은 신경망에게 적은 편입니다

### 그러면 언제 신경망을 쓰나

| 상황 | 권장 |
|---|---|
| 일반적인 표 데이터 | **트리 계열 먼저** (랜덤 포레스트 → 부스팅) |
| 데이터가 아주 많음 (수십만 건 이상) | 신경망도 경쟁력 있음 |
| **표 + 이미지/텍스트가 섞임** | **신경망** (하나의 모델로 통합 가능) |
| 범주가 매우 많음 (사용자 ID 등) | 신경망 (**임베딩**으로 처리) |
| 온라인 학습 (데이터가 계속 들어옴) | 신경망 (`fit`을 이어서 호출 가능) |

**"먼저 랜덤 포레스트로 기준선을 만들고, 그것을 넘지 못하면 신경망을 고집하지 않는다"** 가
표 데이터에서의 합리적인 순서입니다.

> 참고로 이 데이터에서 성능을 더 끌어올리고 싶다면, 신경망 구조를 바꾸는 것보다
> **03번 연습문제 2번에서 확인한 "긴 운행 과소예측" 문제를 해결하는 쪽**이 훨씬 효과적입니다.
> 목적지가 공항인지 알려주는 피처를 추가하는 식으로요. **모델을 바꾸는 것보다
> 피처를 개선하는 것이 대개 더 큰 이득을 줍니다.**

---

## 정리

**1부 — 택시 (회귀)**

- **스케일링은 항상 합니다.** Adam + 비슷한 범위라면 없어도 되지만, SGD를 쓰거나 단위가
  다른 컬럼이 하나만 섞여도 학습이 무너집니다
- **`summary()`로 파라미터 수를 확인**하세요. 학습 데이터 건수와 비슷하면 과적합을 각오해야 합니다
- **학습 곡선이 진단 도구입니다.** 검증 손실이 최저점을 찍고 올라가면 거기서 멈춰야 합니다
- **`EarlyStopping(restore_best_weights=True)`** 로 최적 시점의 가중치를 되찾습니다.
  이 옵션을 빼면 `patience`만큼 나빠진 모델이 남습니다
- **모델을 저장할 때는 스케일러도 함께** 저장해야 합니다

**2부 — 타이타닉 (분류)**

- **은닉층은 그대로, 출력층과 손실 함수만 바꿉니다**

  | 문제 | 출력층 | 손실 함수 |
  |---|---|---|
  | 회귀 | `Dense(1)` | `mse` / `mae` |
  | 이진 분류 | `Dense(1, activation="sigmoid")` | `binary_crossentropy` |
  | 다중 분류 | `Dense(n, activation="softmax")` | `sparse_categorical_crossentropy` |

- **데이터가 적을수록 과적합이 빨리 옵니다.** 429건에서는 10 epoch 만에 검증 손실이 꺾입니다
- **과적합 감시는 `val_loss`로.** 정확도보다 민감하게 반응합니다
- **표 데이터에서는 트리 계열을 먼저 시도하세요.** 회귀·분류 모두 두 모델의 성능이
  사실상 같았는데, 신경망 쪽은 스케일링·구조 설계·과적합 대책이 모두 필요했습니다

## 연습 문제

풀어본 뒤 [04_dnn_keras_solutions.ipynb](04_dnn_keras_solutions.ipynb)에서 확인하세요.
**문제 1~4는 1부(택시·회귀), 문제 5~6은 2부(타이타닉·분류)** 범위입니다.

### 1부 — 택시 (회귀)

**문제 1.** 회귀 모델의 은닉층 구조를 아래 세 가지로 바꿔가며 검증 MAE를 비교하세요.
파라미터 수도 함께 확인하고, **더 큰 모델이 항상 좋은지** 판단해보세요.
1. `Dense(16)` 하나
2. `Dense(64) → Dense(32) → Dense(16)` (본문과 동일)
3. `Dense(256) → Dense(128) → Dense(64) → Dense(32)`

**문제 2.** 회귀 모델의 타깃에 **로그 변환**을 적용해 학습시켜 보세요.
(03번 연습문제 2번에서 제안한 개선안입니다.)
```python
y_train_log = np.log1p(y_train_a)
# 학습 후 예측값은 np.expm1()로 되돌리기
```
전체 MAE와 **40분 이상 구간의 MAE**를 각각 비교하세요. 어느 쪽이 개선되었나요?

**문제 3.** 아래 코드는 학습이 전혀 진행되지 않습니다(손실이 줄지 않음). 원인을 찾아 고치세요.

```python
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_train_s, y_train_a, epochs=30, batch_size=128, verbose=0)
```

**문제 4.** `BatchNormalization`을 넣은 모델과 넣지 않은 모델의 **학습 곡선**을 비교하세요.
수렴 속도와 안정성에 차이가 있나요?

### 2부 — 타이타닉 (분류)

**문제 5.** 이진 분류를 **출력 뉴런 2개 + `softmax`** 로 바꿔 구현하세요.
`to_categorical`로 라벨을 원-핫으로 바꾸고 `categorical_crossentropy`를 써야 합니다.
`sigmoid` 버전과 성능이 비슷하게 나오나요? 두 방식은 수학적으로 어떤 관계인가요?

**문제 6.** `EarlyStopping`에서 `restore_best_weights=True`와 `False`를 각각 적용해
최종 성능을 비교하세요. 차이가 나는 이유를 `patience` 값과 연결해 설명해보세요.

---

이것으로 `tabular-ml-practice` 시리즈가 끝납니다. 전체 흐름을 다시 정리하려면
[시리즈 README](../README.md)를 참고하세요.